# PatchTST Stock Classifier for S&P 500 Stocks

This notebook mirrors the reference PatchTST pipeline, but adapts it for a classifier: for each context window, predict whether each of the next five trading days closes significantly lower, roughly flat, or significantly higher.

The notebook compares three tracks:

- IBM Granite PatchTST zero-shot forecast baseline, then threshold forecasts into classes.
- A from-scratch global PatchTST classifier trained across all tickers.
- From-scratch per-ticker PatchTST classifiers.

Class IDs: `0 = down`, `1 = flat`, `2 = up`.

## 0. Environment Setup

Install from `requirements.txt` in a virtual environment or uncomment the `%pip` line below for a notebook-local install.

`peft` is included for parity with the IBM/LoRA reference workflow, but LoRA is not used by the from-scratch classifier. `bitsandbytes` is CUDA/Linux-only and is intentionally commented in `requirements.txt`.

In [ ]:
# Uncomment to install inside the active notebook kernel.
# %pip install -r requirements.txt

import platform
from pathlib import Path

import torch

# ---- DEVICE / DTYPE CONFIG (override here to force a backend) ----
DEVICE_OVERRIDE = None   # None | 'cuda' | 'mps' | 'cpu'
DTYPE_OVERRIDE = None    # None | 'bfloat16' | 'float16' | 'float32'


def _autoselect_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def _autoselect_dtype(device):
    if device.type == 'cuda':
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    # MPS bf16 is flaky; CPU bf16 is slow and many ops fall back to fp32.
    return torch.float32


device = torch.device(DEVICE_OVERRIDE) if DEVICE_OVERRIDE else _autoselect_device()
dtype = getattr(torch, DTYPE_OVERRIDE) if DTYPE_OVERRIDE else _autoselect_dtype(device)
NUM_WORKERS = 0 if platform.system() == 'Windows' else 4

print(f'OS={platform.system()} | device={device} | dtype={dtype} | workers={NUM_WORKERS}')

In [ ]:
import inspect
import os
import sys
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch.utils.data import DataLoader
from transformers import EarlyStoppingCallback, PatchTSTConfig, TrainingArguments, Trainer

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebook':
    # Supports running from the repo root in VS Code/Cursor.
    NOTEBOOK_DIR = Path('models/notebook').resolve()

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DATA_RAW_DIR = PROJECT_ROOT / 'models' / 'data_raw'
CHECKPOINT_DIR = NOTEBOOK_DIR / 'checkpoint' / 'patchtst_cls'
SAVE_DIR = NOTEBOOK_DIR / 'save_dir'

sys.path.insert(0, str(NOTEBOOK_DIR))

from classification_head import PatchTSTClassifier
from dataset_utils import (
    StockWindowClassificationDataset,
    compute_class_weights,
    iter_tickers,
    split_by_fraction,
    summarize_dataset,
)
from labeling import CLASS_NAMES, LabelConfig, make_class_labels

sns.set_theme(style='whitegrid')

## 1. Notebook Configuration

The first block is the IBM Granite-compatible reference config. It stays commented because it is only used in the zero-shot baseline section.

The active block configures the from-scratch classifier. Labeling starts with a fixed ±1% threshold, but `LABEL_RULE` can later be changed to `rolling_vol` or `atr` without changing the model head.

In [ ]:
# ---- IBM Granite-compatible config (IBM baseline only) ----
# GRANITE_BASE_MODEL = 'ibm-granite/granite-timeseries-patchtst'
# GRANITE_CONTEXT_LENGTH = 512
# GRANITE_PREDICTION_LENGTH = 96
# GRANITE_TARGET_COLUMNS = ['Close']

# ---- From-scratch classifier config ----
TIMESTAMP_COLUMN = 'Date'
TICKER_COLUMN = 'Ticker'
TARGET_COLUMNS = ['Open', 'High', 'Low', 'Close', 'Volume']
ID_COLUMNS = ['Ticker']

CONTEXT_LENGTH = 128       # roughly six months of trading days
FORECAST_HORIZON = 5       # one trading week
PATCH_LENGTH = 8
PATCH_STRIDE = 8
BATCH_SIZE = 64
TRAIN_FRAC = 0.8
VALID_FRAC = 0.1
DATASET_NAME = 'sp500_daily'

# ---- Labeling hyperparameters ----
LABEL_RULE = 'fixed_pct'   # 'fixed_pct' | 'rolling_vol' | 'atr'
LABEL_THRESHOLD = 0.01     # 1% threshold for fixed_pct
VOL_WINDOW = 21
VOL_K = 0.5
N_CLASSES = 3

# ---- Training controls ----
GLOBAL_NUM_EPOCHS = 10
PER_TICKER_NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
EARLY_STOPPING_PATIENCE = 3
RUN_GRANITE_BASELINE = True
RUN_GLOBAL_TRAINING = True
RUN_PER_TICKER_TRAINING = True
MAX_TICKERS_FOR_PER_TICKER = None  # set to an int for a quick smoke test

label_config = LabelConfig(
    rule=LABEL_RULE,
    threshold=LABEL_THRESHOLD,
    vol_window=VOL_WINDOW,
    vol_k=VOL_K,
)

print(label_config)

## 2. Load & Preprocess CSV

The raw files already use the needed long format: `Date, Ticker, Open, High, Low, Close, Volume`.

Volume is log-transformed before scaling because it is orders of magnitude larger than OHLC prices. The notebook attempts to use IBM Granite TSFM's `TimeSeriesPreprocessor`; if that import/API is unavailable, it falls back to explicit per-ticker z-score scaling so the rest of the pipeline still runs.

In [ ]:
csv_paths = sorted(DATA_RAW_DIR.glob('*_raw.csv'))
if not csv_paths:
    raise FileNotFoundError(f'No raw CSV files found in {DATA_RAW_DIR}')

raw_df = pd.concat(
    [pd.read_csv(path, parse_dates=[TIMESTAMP_COLUMN]) for path in csv_paths],
    ignore_index=True,
)
raw_df = raw_df.sort_values([TICKER_COLUMN, TIMESTAMP_COLUMN]).reset_index(drop=True)
raw_df[TICKER_COLUMN] = raw_df[TICKER_COLUMN].astype(str)
raw_df['Volume'] = np.log1p(raw_df['Volume'].astype(float))

print(raw_df.head())
print(raw_df.groupby(TICKER_COLUMN).size())

# Sample weekly-ish context plot, matching the spirit of the reference notebook.
sample_ticker = raw_df[TICKER_COLUMN].iloc[0]
sample = raw_df[raw_df[TICKER_COLUMN] == sample_ticker].tail(CONTEXT_LENGTH + FORECAST_HORIZON)
plt.figure(figsize=(12, 4))
plt.plot(sample[TIMESTAMP_COLUMN], sample['Close'], label=f'{sample_ticker} Close')
plt.axvline(sample[TIMESTAMP_COLUMN].iloc[-FORECAST_HORIZON], color='tab:red', linestyle='--', label='forecast start')
plt.title(f'{sample_ticker} sample context + one-week forecast horizon')
plt.xlabel('Date')
plt.ylabel('Close')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def manual_per_ticker_standardize(train_df, valid_df, test_df, target_columns, ticker_column='Ticker'):
    stats = {}
    for ticker, group in train_df.groupby(ticker_column):
        mean = group[target_columns].astype(float).mean()
        std = group[target_columns].astype(float).std().replace(0, 1.0).fillna(1.0)
        stats[str(ticker)] = (mean, std)

    global_mean = train_df[target_columns].astype(float).mean()
    global_std = train_df[target_columns].astype(float).std().replace(0, 1.0).fillna(1.0)

    def transform(frame):
        frame = frame.copy()
        for ticker, idx in frame.groupby(ticker_column).groups.items():
            mean, std = stats.get(str(ticker), (global_mean, global_std))
            frame.loc[idx, target_columns] = (frame.loc[idx, target_columns].astype(float) - mean) / std
        return frame

    return transform(train_df), transform(valid_df), transform(test_df), stats


def try_granite_preprocess(train_df, valid_df, test_df):
    try:
        from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
    except Exception as exc:
        print(f'Granite TimeSeriesPreprocessor unavailable; using manual scaler. Reason: {exc}')
        return (*manual_per_ticker_standardize(train_df, valid_df, test_df, TARGET_COLUMNS, TICKER_COLUMN), None)

    try:
        tsp = TimeSeriesPreprocessor(
            timestamp_column=TIMESTAMP_COLUMN,
            id_columns=ID_COLUMNS,
            target_columns=TARGET_COLUMNS,
            scaling='std',
        )
        tsp.train(train_df)

        def transform(frame):
            if hasattr(tsp, 'preprocess'):
                return tsp.preprocess(frame)
            if hasattr(tsp, 'transform'):
                return tsp.transform(frame)
            return tsp(frame)

        return transform(train_df), transform(valid_df), transform(test_df), None, tsp
    except Exception as exc:
        print(f'Granite preprocessing failed; using manual scaler. Reason: {exc}')
        return (*manual_per_ticker_standardize(train_df, valid_df, test_df, TARGET_COLUMNS, TICKER_COLUMN), None)


train_raw, valid_raw, test_raw = split_by_fraction(
    raw_df,
    train_frac=TRAIN_FRAC,
    valid_frac=VALID_FRAC,
    timestamp_column=TIMESTAMP_COLUMN,
    ticker_column=TICKER_COLUMN,
    context_length=CONTEXT_LENGTH,
)

train_values, valid_values, test_values, scaler_stats, tsp = try_granite_preprocess(train_raw, valid_raw, test_raw)

train_ds = StockWindowClassificationDataset(train_values, train_raw, TARGET_COLUMNS, CONTEXT_LENGTH, FORECAST_HORIZON, label_config)
valid_ds = StockWindowClassificationDataset(valid_values, valid_raw, TARGET_COLUMNS, CONTEXT_LENGTH, FORECAST_HORIZON, label_config)
test_ds = StockWindowClassificationDataset(test_values, test_raw, TARGET_COLUMNS, CONTEXT_LENGTH, FORECAST_HORIZON, label_config)

dataset_summary = pd.concat([
    summarize_dataset('train', train_ds),
    summarize_dataset('valid', valid_ds),
    summarize_dataset('test', test_ds),
], ignore_index=True)
dataset_summary

### Optional Granite Padding

Granite's published PatchTST checkpoint expects a longer context than the from-scratch classifier. Use this only in the IBM baseline path when a series does not have enough context. For the included raw datasets, each ticker has enough history.

In [ ]:
# Optional, IBM-config only:
# from ts_padding_utils import pad_batch_for_granite
# padded_batch = pad_batch_for_granite(batch, context_length=512)

## 3-4. Label Generation, Metrics, and Visualization

Labels are generated inside `StockWindowClassificationDataset` from original close prices. The model sees scaled OHLCV values, but the class target is always based on the unscaled close move from the final context day to each future day.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)

    flat_preds = preds.reshape(-1)
    flat_labels = labels.reshape(-1)

    metrics = {
        'accuracy': accuracy_score(flat_labels, flat_preds),
        'macro_f1': f1_score(flat_labels, flat_preds, average='macro', labels=[0, 1, 2], zero_division=0),
    }

    for day_idx in range(labels.shape[1]):
        metrics[f'day_{day_idx + 1}_accuracy'] = accuracy_score(labels[:, day_idx], preds[:, day_idx])

    per_class = f1_score(flat_labels, flat_preds, average=None, labels=[0, 1, 2], zero_division=0)
    for class_id, score in enumerate(per_class):
        metrics[f'{CLASS_NAMES[class_id]}_f1'] = score

    return metrics


def confusion_matrices_by_day(labels, preds):
    matrices = []
    for day_idx in range(labels.shape[1]):
        matrices.append(confusion_matrix(labels[:, day_idx], preds[:, day_idx], labels=[0, 1, 2]))
    return matrices


def evaluate_logits(logits, labels, name='model'):
    preds = np.argmax(logits, axis=-1)
    metrics = compute_metrics((logits, labels))
    print(f"{name} accuracy={metrics['accuracy']:.4f} macro_f1={metrics['macro_f1']:.4f}")
    return metrics, preds

In [ ]:
def trainer_predict_logits(trainer, dataset):
    output = trainer.predict(dataset)
    logits = output.predictions[0] if isinstance(output.predictions, tuple) else output.predictions
    return logits, output.label_ids


def evaluate_and_visualize(model_or_trainer, dataset, name='model', sample_idx=0):
    if isinstance(model_or_trainer, Trainer):
        logits, labels = trainer_predict_logits(model_or_trainer, dataset)
    else:
        loader = DataLoader(dataset, batch_size=BATCH_SIZE)
        logits_batches, label_batches = [], []
        model_or_trainer.eval()
        model_or_trainer.to(device)
        with torch.no_grad():
            for batch in loader:
                past_values = batch['past_values'].to(device)
                outputs = model_or_trainer(past_values=past_values)
                logits_batches.append(outputs.logits.detach().cpu().numpy())
                label_batches.append(batch['labels'].numpy())
        logits = np.concatenate(logits_batches, axis=0)
        labels = np.concatenate(label_batches, axis=0)

    metrics, preds = evaluate_logits(logits, labels, name=name)

    fig, axes = plt.subplots(1, FORECAST_HORIZON, figsize=(4 * FORECAST_HORIZON, 3), sharey=True)
    if FORECAST_HORIZON == 1:
        axes = [axes]
    matrices = confusion_matrices_by_day(labels, preds)
    for day_idx, (ax, matrix) in enumerate(zip(axes, matrices), start=1):
        sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                    xticklabels=['down', 'flat', 'up'], yticklabels=['down', 'flat', 'up'])
        ax.set_title(f'Day {day_idx}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    fig.suptitle(f'{name} confusion matrices')
    plt.tight_layout()
    plt.show()

    per_day = [metrics[f'day_{idx + 1}_accuracy'] for idx in range(FORECAST_HORIZON)]
    plt.figure(figsize=(8, 4))
    sns.barplot(x=[f'Day {idx + 1}' for idx in range(FORECAST_HORIZON)], y=per_day)
    plt.ylim(0, 1)
    plt.title(f'{name} per-day accuracy')
    plt.ylabel('Accuracy')
    plt.tight_layout()
    plt.show()

    sample_idx = min(sample_idx, len(labels) - 1)
    sample_meta = dataset.metadata(sample_idx) if hasattr(dataset, 'metadata') else None
    sample_df = pd.DataFrame({
        'day': [f'Day {idx + 1}' for idx in range(FORECAST_HORIZON)],
        'actual': [CLASS_NAMES[int(x)] for x in labels[sample_idx]],
        'predicted': [CLASS_NAMES[int(x)] for x in preds[sample_idx]],
    })
    if sample_meta:
        print(sample_meta)
    display(sample_df)

    return {'metrics': metrics, 'predictions': preds, 'labels': labels, 'logits': logits}

## 5. IBM Granite Zero-Shot Baseline (IBM-Config Only)

This section uses `ibm-granite/granite-timeseries-patchtst` as a domain-shifted zero-shot forecaster. Granite was not trained on stock OHLCV data, so this is a baseline comparison rather than the primary model.

The model forecasts continuous closes. We convert the first five forecasted closes into up/flat/down classes using the exact same label rule as the trained classifier.

In [ ]:
def _prediction_array(outputs):
    for attr in ('prediction_outputs', 'prediction_logits', 'logits'):
        if hasattr(outputs, attr) and getattr(outputs, attr) is not None:
            value = getattr(outputs, attr)
            return value[0] if isinstance(value, tuple) else value
    if isinstance(outputs, tuple):
        return outputs[0]
    raise RuntimeError('Could not locate prediction tensor in Granite output.')


def run_granite_zero_shot_baseline():
    from transformers import PatchTSTForPrediction

    GRANITE_BASE_MODEL = 'ibm-granite/granite-timeseries-patchtst'
    GRANITE_CONTEXT_LENGTH = 512
    GRANITE_TARGET_COLUMNS = ['Close']

    granite_test = StockWindowClassificationDataset(
        values_df=test_raw[[TIMESTAMP_COLUMN, TICKER_COLUMN, 'Close']].copy(),
        label_df=test_raw.copy(),
        target_columns=GRANITE_TARGET_COLUMNS,
        context_length=GRANITE_CONTEXT_LENGTH,
        forecast_horizon=FORECAST_HORIZON,
        label_config=label_config,
    )
    print(f'Granite baseline windows: {len(granite_test)}')

    model = PatchTSTForPrediction.from_pretrained(GRANITE_BASE_MODEL).to(device)
    model.eval()

    all_preds, all_labels = [], []
    loader = DataLoader(granite_test, batch_size=min(BATCH_SIZE, 32), num_workers=NUM_WORKERS)
    with torch.no_grad():
        for batch in loader:
            past_values = batch['past_values'].to(device=device, dtype=torch.float32)
            outputs = model(past_values=past_values)
            forecasts = _prediction_array(outputs).detach().cpu().numpy()
            forecasts = forecasts[:, :FORECAST_HORIZON, 0] if forecasts.ndim == 3 else forecasts[:, :FORECAST_HORIZON]
            anchors = batch['past_values'][:, -1, 0].detach().cpu().numpy()
            pred_labels = make_class_labels(
                future_close=forecasts,
                past_close_t=anchors,
                rule=LABEL_RULE,
                threshold=LABEL_THRESHOLD,
                vol_window=VOL_WINDOW,
                vol_k=VOL_K,
            )
            all_preds.append(pred_labels)
            all_labels.append(batch['labels'].numpy())

    preds = np.concatenate(all_preds, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    logits = np.eye(N_CLASSES)[preds]
    metrics, _ = evaluate_logits(logits, labels, name='IBM Granite zero-shot')
    return {'metrics': metrics, 'predictions': preds, 'labels': labels, 'logits': logits}


granite_results = None
if RUN_GRANITE_BASELINE:
    try:
        granite_results = run_granite_zero_shot_baseline()
    except Exception as exc:
        print(f'Granite baseline skipped: {exc}')
else:
    print('Granite baseline disabled by RUN_GRANITE_BASELINE=False')

## 6. From-Scratch PatchTST Classifier — Global Model

This model trains across all tickers using the same PatchTST encoder family, but with a custom multi-day classification head instead of the forecasting head.

TensorBoard logs are written under `./checkpoint/patchtst_cls/global/logs/...`.

In [ ]:
def make_patchtst_config(context_length=CONTEXT_LENGTH, num_input_channels=None):
    return PatchTSTConfig(
        do_mask_input=False,
        context_length=context_length,
        patch_length=PATCH_LENGTH,
        patch_stride=PATCH_STRIDE,
        num_input_channels=num_input_channels or len(TARGET_COLUMNS),
        prediction_length=FORECAST_HORIZON,
        d_model=128,
        num_attention_heads=16,
        num_hidden_layers=3,
        ffn_dim=512,
        dropout=0.2,
        head_dropout=0.2,
        pooling_type=None,
        channel_attention=False,
        scaling='std',
        loss='ce',
        pre_norm=True,
        norm_type='batchnorm',
    )


def make_training_args(output_dir, logging_dir, num_epochs):
    params = inspect.signature(TrainingArguments.__init__).parameters
    kwargs = dict(
        output_dir=str(output_dir),
        overwrite_output_dir=True,
        learning_rate=LEARNING_RATE,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        dataloader_num_workers=NUM_WORKERS,
        save_strategy='epoch',
        logging_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='eval_macro_f1',
        greater_is_better=True,
        logging_dir=str(logging_dir),
        report_to='tensorboard',
        label_names=['labels'],
        remove_unused_columns=False,
        fp16=(dtype == torch.float16 and device.type == 'cuda'),
        bf16=(dtype == torch.bfloat16 and device.type == 'cuda'),
    )
    kwargs['eval_strategy' if 'eval_strategy' in params else 'evaluation_strategy'] = 'epoch'
    if 'use_cpu' in params:
        kwargs['use_cpu'] = device.type == 'cpu'
    elif 'no_cuda' in params:
        kwargs['no_cuda'] = device.type != 'cuda'
    return TrainingArguments(**kwargs)


class_weights = compute_class_weights(train_ds).to(device)
print('Class weights:', class_weights.detach().cpu().numpy())

global_config = make_patchtst_config(num_input_channels=len(TARGET_COLUMNS))
global_model = PatchTSTClassifier(
    config=global_config,
    horizon=FORECAST_HORIZON,
    n_classes=N_CLASSES,
    class_weights=class_weights,
).to(device=device, dtype=dtype if device.type == 'cuda' else torch.float32)

global_output_dir = CHECKPOINT_DIR / 'global' / 'output'
global_logging_dir = CHECKPOINT_DIR / 'global' / 'logs' / datetime.now().strftime('%Y%m%d_%H%M%S')

global_trainer = Trainer(
    model=global_model,
    args=make_training_args(global_output_dir, global_logging_dir, GLOBAL_NUM_EPOCHS),
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

global_results = None
if RUN_GLOBAL_TRAINING:
    print(f'Training global PatchTST classifier on {DATASET_NAME}')
    global_trainer.train()
    global_results = evaluate_and_visualize(global_trainer, test_ds, name='global PatchTST classifier')
else:
    print('Global training disabled by RUN_GLOBAL_TRAINING=False')

In [ ]:
# Sanity check: prediction logits should be (num_windows, 5, 3).
if RUN_GLOBAL_TRAINING and global_results is not None:
    print(global_results['logits'].shape)
else:
    sample_batch = next(iter(DataLoader(test_ds, batch_size=4)))
    global_model.eval()
    with torch.no_grad():
        sample_outputs = global_model(past_values=sample_batch['past_values'].to(device))
    print(sample_outputs.logits.shape)

## 7. From-Scratch PatchTST Classifier — Per-Ticker Models

This section trains one classifier per ticker. It is useful for comparison, but it has much less data per model and can be slow on CPU.

Set `MAX_TICKERS_FOR_PER_TICKER` in the config cell for a quick smoke test.

In [ ]:
def build_single_ticker_datasets(ticker):
    ticker_df = raw_df[raw_df[TICKER_COLUMN] == ticker].copy()
    tr_raw, va_raw, te_raw = split_by_fraction(
        ticker_df,
        train_frac=TRAIN_FRAC,
        valid_frac=VALID_FRAC,
        timestamp_column=TIMESTAMP_COLUMN,
        ticker_column=TICKER_COLUMN,
        context_length=CONTEXT_LENGTH,
    )
    tr_values, va_values, te_values, _, _ = try_granite_preprocess(tr_raw, va_raw, te_raw)
    return (
        StockWindowClassificationDataset(tr_values, tr_raw, TARGET_COLUMNS, CONTEXT_LENGTH, FORECAST_HORIZON, label_config),
        StockWindowClassificationDataset(va_values, va_raw, TARGET_COLUMNS, CONTEXT_LENGTH, FORECAST_HORIZON, label_config),
        StockWindowClassificationDataset(te_values, te_raw, TARGET_COLUMNS, CONTEXT_LENGTH, FORECAST_HORIZON, label_config),
    )


def train_one_ticker(ticker):
    ticker_train, ticker_valid, ticker_test = build_single_ticker_datasets(ticker)
    if not len(ticker_train) or not len(ticker_valid) or not len(ticker_test):
        print(f'Skipping {ticker}: not enough windows')
        return None

    weights = compute_class_weights(ticker_train).to(device)
    model = PatchTSTClassifier(
        config=make_patchtst_config(num_input_channels=len(TARGET_COLUMNS)),
        horizon=FORECAST_HORIZON,
        n_classes=N_CLASSES,
        class_weights=weights,
    ).to(device=device, dtype=dtype if device.type == 'cuda' else torch.float32)

    output_dir = CHECKPOINT_DIR / 'per_ticker' / ticker / 'output'
    logging_dir = CHECKPOINT_DIR / 'per_ticker' / ticker / 'logs' / datetime.now().strftime('%Y%m%d_%H%M%S')
    trainer = Trainer(
        model=model,
        args=make_training_args(output_dir, logging_dir, PER_TICKER_NUM_EPOCHS),
        train_dataset=ticker_train,
        eval_dataset=ticker_valid,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    print(f'Training per-ticker model for {ticker}')
    trainer.train()
    logits, labels = trainer_predict_logits(trainer, ticker_test)
    metrics, preds = evaluate_logits(logits, labels, name=f'{ticker} PatchTST classifier')
    return {
        'ticker': ticker,
        'trainer': trainer,
        'metrics': metrics,
        'predictions': preds,
        'labels': labels,
        'logits': logits,
    }


per_ticker_results = []
if RUN_PER_TICKER_TRAINING:
    ticker_list = list(iter_tickers(raw_df, TICKER_COLUMN))
    if MAX_TICKERS_FOR_PER_TICKER is not None:
        ticker_list = ticker_list[:MAX_TICKERS_FOR_PER_TICKER]
    for ticker in ticker_list:
        result = train_one_ticker(ticker)
        if result is not None:
            per_ticker_results.append(result)
else:
    print('Per-ticker training disabled by RUN_PER_TICKER_TRAINING=False')

len(per_ticker_results)

## 8. Comparison & Visualization

Compare IBM Granite zero-shot, the global classifier, and the average per-ticker classifier performance.

In [ ]:
comparison_rows = []

if granite_results is not None:
    comparison_rows.append({
        'model': 'IBM Granite zero-shot',
        'accuracy': granite_results['metrics']['accuracy'],
        'macro_f1': granite_results['metrics']['macro_f1'],
        'kind': 'baseline',
    })

if global_results is not None:
    comparison_rows.append({
        'model': 'Global PatchTST classifier',
        'accuracy': global_results['metrics']['accuracy'],
        'macro_f1': global_results['metrics']['macro_f1'],
        'kind': 'trained_global',
    })

for result in per_ticker_results:
    comparison_rows.append({
        'model': f"{result['ticker']} per-ticker",
        'accuracy': result['metrics']['accuracy'],
        'macro_f1': result['metrics']['macro_f1'],
        'kind': 'trained_per_ticker',
    })

comparison_df = pd.DataFrame(comparison_rows)
if comparison_df.empty:
    print('No completed model results yet. Run at least one baseline/training section first.')
else:
    summary_rows = []
    for kind, group in comparison_df.groupby('kind'):
        summary_rows.append({
            'model_group': kind,
            'accuracy_mean': group['accuracy'].mean(),
            'accuracy_min': group['accuracy'].min(),
            'accuracy_max': group['accuracy'].max(),
            'macro_f1_mean': group['macro_f1'].mean(),
            'macro_f1_min': group['macro_f1'].min(),
            'macro_f1_max': group['macro_f1'].max(),
            'n_models': len(group),
        })
    comparison_summary = pd.DataFrame(summary_rows)
    display(comparison_summary)

    plot_df = comparison_summary.melt(
        id_vars=['model_group'],
        value_vars=['accuracy_mean', 'macro_f1_mean'],
        var_name='metric',
        value_name='score',
    )
    plt.figure(figsize=(10, 5))
    sns.barplot(data=plot_df, x='model_group', y='score', hue='metric')
    plt.ylim(0, 1)
    plt.title('PatchTST stock direction classifier comparison')
    plt.xlabel('Model group')
    plt.ylabel('Score')
    plt.tight_layout()
    comparison_plot_path = CHECKPOINT_DIR / 'model_comparison.png'
    comparison_plot_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(comparison_plot_path, dpi=160)
    plt.show()
    print(f'Saved comparison plot to {comparison_plot_path}')

## 9. Save Model/Adapter & Optional Push to Hub

The from-scratch model is saved locally. Publishing to Hugging Face is optional and remains commented out.

In [ ]:
global_save_dir = SAVE_DIR / 'patchtst_cls_global'
global_save_dir.mkdir(parents=True, exist_ok=True)

if global_model is not None:
    torch.save(global_model.state_dict(), global_save_dir / 'pytorch_model.bin')
    global_config.save_pretrained(global_save_dir)
    pd.Series({
        'context_length': CONTEXT_LENGTH,
        'forecast_horizon': FORECAST_HORIZON,
        'label_rule': LABEL_RULE,
        'label_threshold': LABEL_THRESHOLD,
        'target_columns': ','.join(TARGET_COLUMNS),
    }).to_json(global_save_dir / 'training_metadata.json', indent=2)
    print('Model saved to', global_save_dir.resolve())

# Optional - requires `huggingface-cli login` or `from huggingface_hub import login; login()`.
# Because PatchTSTClassifier is a local wrapper class, prefer uploading the full
# folder with README + source files rather than calling model.push_to_hub directly.
# from huggingface_hub import HfApi
# HfApi().upload_folder(
#     folder_path=str(global_save_dir),
#     repo_id='your-username/patchtst-stock-classifier',
#     repo_type='model',
# )

## 10. Re-Evaluate Fine-Tuned Model

Reload the saved global classifier from disk and re-run the evaluation visualization to verify the checkpoint works.

In [ ]:
reloaded_results = None
checkpoint_path = global_save_dir / 'pytorch_model.bin'
if checkpoint_path.exists():
    reloaded_config = PatchTSTConfig.from_pretrained(global_save_dir)
    reloaded_model = PatchTSTClassifier(
        config=reloaded_config,
        horizon=FORECAST_HORIZON,
        n_classes=N_CLASSES,
        class_weights=class_weights.detach().cpu(),
    )
    # Materialize LazyLinear before loading the saved classifier head weights.
    sample_batch = next(iter(DataLoader(test_ds, batch_size=1)))
    with torch.no_grad():
        _ = reloaded_model(past_values=sample_batch['past_values'])
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    reloaded_model.load_state_dict(state_dict)
    reloaded_model.to(device=device, dtype=dtype if device.type == 'cuda' else torch.float32)
    reloaded_results = evaluate_and_visualize(reloaded_model, test_ds, name='reloaded global PatchTST classifier')
else:
    print(f'No saved checkpoint found at {checkpoint_path}. Run the save cell first.')

## TensorBoard

Run this from `models/notebook` after training starts:

```bash
tensorboard --logdir ./checkpoint/patchtst_cls/
```

Then open the printed local URL in your browser.